# 01 – Data Exploration
Load the Resume-JD Match dataset, check its structure, and split each row into JD and resume.

## Install

In [ ]:
!pip install -q datasets

## Load dataset
6,241 train and 1,759 test rows; columns: text, label.

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("facehuggerapoorv/resume-jd-match")
print(ds)
print(ds["train"].features)

train = ds["train"].to_pandas()
test = ds["test"].to_pandas()
print(train.shape, test.shape)

README.md:   0%|          | 0.00/411 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 29.0MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 8.07MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1759 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 6241
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1759
    })
})
{'text': Value('string'), 'label': Value('string')}
(6241, 2) (1759, 2)


## Label distribution

In [ ]:
print(train["label"].value_counts().sort_index())
print(test["label"].value_counts().sort_index())

label
Good Fit         1542
No Fit           3143
Potential Fit    1556
Name: count, dtype: int64
label
Good Fit         458
No Fit           857
Potential Fit    444
Name: count, dtype: int64


## Raw records
Read sample records to understand the text format.

In [ ]:
for i in range(2):
    print(f"===== Record {i} | label = {train.loc[i, 'label']} =====")
    print(train.loc[i, "text"][:2000])
    print()

===== Record 0 | label = No Fit =====
For the given job description <<Net2Source Inc. is an award-winning total workforce solutions company recognized by Staffing Industry Analysts for our accelerated growth of 300% in the last 3 years with over 5500+ employees globally, with over 30+ locations in the US and global operations in 32 countries. We believe in providing staffing solutions to address the current talent gap  Right Talent  Right Time  Right Place  Right Price and acting as a Career Coach to our consultants.  
Role: Basel Business AnalystLocation: Washington, D.C.Work Mode: HybridHire Type: 6+ Month Contract (extendable)
JD: Role Specific Experience: 6+ years of relevant technical and business work experience The Candidates who have worked on Basel related projects in Credit risks or at least are aware of credit risk.  Banking & Financial domain experience, along with knowledge of risk management, familiarity with concepts of finance and accounting  Proficiency in MS Excel  Ex

## Find the JD/resume boundary

In [ ]:
sample = train.loc[0, "text"]
print("'<<' at:", [i for i in range(len(sample)) if sample.startswith("<<", i)])
print("'>>' at:", [i for i in range(len(sample)) if sample.startswith(">>", i)])
print(sample[-800:])

'<<' at: [30, 4005]
'>>' at: [3990, 10705]
gs, Contracts, Customer Service, Direct Sales, Fixed Assets, Food Safety, Government Contracts, Logistics, Maintenance, Million, Quality Assurance, Receptionist, Retail Sales, Sales And, Shipping, Usda, Accounts Payable, Solutions, Staffing, Cash, Clerk, Merchandising, Ordering, Automated Payroll, Payroll Processing, Software Training, Training, Accountant, The Accounts, Accounts Receivable, Credit, Billing, Csi, Excel, Ms Excel, Networking, All Sales, Basis, Cash Management, Forecasting, Gl, Telephone, Adjustments, Audit, Balance Sheet, Budgeting, Monthly Sales, Sales Tax, Annuities, Cpa, Life Insurance, Procedure Writing, Project Management, Public Relations, Team Building, Administrative Manager, Entrees, Entrées, Hr, Microsoft Office, Ms Office, Purchase Orders>>. The result is, No Fit


## Length, duplicates and leakage

In [ ]:
print(train["text"].str.len().describe())
print("Duplicates in train:", train["text"].duplicated().sum())
print("Duplicates in test:", test["text"].duplicated().sum())
print("Test rows also in train (leakage):", test["text"].isin(train["text"]).sum())

count     6241.000000
mean      8611.970678
std       3428.782271
min       1497.000000
25%       6329.000000
50%       7953.000000
75%      10166.000000
max      28739.000000
Name: text, dtype: float64
Duplicates in train: 1
Duplicates in test: 0
Test rows also in train (leakage): 0


## Confirm the pattern on all rows
Every row has the same structure, and the label is embedded in the training text.

In [ ]:
import re

s = train.loc[0, "text"]
print("Start:", repr(s[:32]))
print("Between JD and resume:", repr(s[3990:4007]))

print("\n'<<' count per row:\n", train["text"].str.count("<<").value_counts())
print("\n'>>' count per row:\n", train["text"].str.count(">>").value_counts())

mid = train["text"].str.extract(r">>(.*?)<<", flags=re.S)[0]
print("\nText between JD and resume (top 5):\n", mid.value_counts().head())

tail = train["text"].str.extract(r"The result is,\s*(.+?)\s*$")[0]
print("\nRows with 'The result is':", tail.notna().sum(), "of", len(train))
print("Embedded label matches label column:", (tail == train["label"]).mean())

Start: 'For the given job description <<'
Between JD and resume: '>> the resume: <<'

'<<' count per row:
 text
2    6241
Name: count, dtype: int64

'>>' count per row:
 text
2    6241
Name: count, dtype: int64

Text between JD and resume (top 5):
 0
the resume:     6241
Name: count, dtype: int64

Rows with 'The result is': 6241 of 6241
Embedded label matches label column: 1.0


## Split JD and resume, remove embedded label
Train ends with "The result is, <label>"; test ends with "The result is," and no label. The embedded label is removed.

In [ ]:
pattern = r"^For the given job description <<(.*)>> the resume: <<(.*)>>\. The result is,\s*(.*?)\s*$"

def split_text(df):
    parts = df["text"].str.extract(pattern, flags=re.S)
    parts.columns = ["jd", "resume", "embedded_label"]
    return df.join(parts)

train_s = split_text(train)
test_s = split_text(test)

for name, d in [("train", train_s), ("test", test_s)]:
    print(f"--- {name} ---")
    print("Unparsed rows:", d["jd"].isna().sum())
    print("Embedded label present:", (d["embedded_label"] != "").sum(), "of", len(d))
    print("Embedded label matches (where present):",
          (d.loc[d["embedded_label"] != "", "embedded_label"] == d.loc[d["embedded_label"] != "", "label"]).mean())
    print("Empty JD:", (d["jd"].str.strip() == "").sum(), "| Empty resume:", (d["resume"].str.strip() == "").sum())
    print("Leftover 'The result is' in resume:", d["resume"].str.contains("The result is").sum())
    print(d[["jd", "resume"]].apply(lambda c: c.str.len()).describe().loc[["mean", "min", "max"]])

print("\nResume sample:\n", train_s.loc[0, "resume"][:500])

--- train ---
Unparsed rows: 0
Embedded label present: 6241 of 6241
Embedded label matches (where present): 1.0
Empty JD: 0 | Empty resume: 0
Leftover 'The result is' in resume: 0
               jd       resume
mean  2731.328313   5804.40298
min     72.000000    897.00000
max   7669.000000  25364.00000
--- test ---
Unparsed rows: 0
Embedded label present: 0 of 1759
Embedded label matches (where present): nan
Empty JD: 0 | Empty resume: 0
Leftover 'The result is' in resume: 0
               jd        resume
mean  2939.185333   5663.259238
min    238.000000    897.000000
max   8171.000000  25364.000000

Resume sample:
 SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market.
Highlights-Soft Skills: Public Speaking, Public Relations, Team Building, Project Management, Procedure writing, Staff Supervisio

## Save to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
out_dir = "/content/drive/MyDrive/resume-jd-matching/data"
os.makedirs(out_dir, exist_ok=True)

train_clean = train_s[["jd", "resume", "label"]]
test_clean = test_s[["jd", "resume", "label"]]
train_clean.to_parquet(f"{out_dir}/train_split.parquet", index=False)
test_clean.to_parquet(f"{out_dir}/test_split.parquet", index=False)
print("Saved:", train_clean.shape, test_clean.shape)

Mounted at /content/drive
Saved: (6241, 3) (1759, 3)
